# Experimenting with step size for a galaxy smaller than the ellipse

In [13]:
# imports
from importlib import reload
import os
from importlib.resources import files as resource_files

import numpy as np

import pandas

from astropy.coordinates import SkyCoord
from astropy.coordinates import offset_by
from astropy import units
#from astropy.io import fits

from astropath import path
from astropath import localization
from astropath import bayesian

# Convenience class

In [14]:
Path = path.PATH()

# Faux FRB

In [15]:
## FRB Coord
frb_coord = SkyCoord('21h44m25.255s -40d54m00.10s', frame='icrs')
frb_coord

<SkyCoord (ICRS): (ra, dec) in deg
    (326.10522917, -40.90002778)>

# Prior

In [16]:
theta_prior = dict(max=6., PDF='exp', scale=1.)

# Localization

In [17]:
# Small, but not tiny
eellipse = dict(a=5, b=5, theta=0.)

## Build it

In [18]:
Path.init_localization('eellipse', center_coord=frb_coord, eellipse=eellipse)

In [19]:
Path.localiz

{'type': 'eellipse',
 'center_coord': <SkyCoord (ICRS): (ra, dec) in deg
     (326.10522917, -40.90002778)>,
 'eellipse': {'a': 5, 'b': 5, 'theta': 0.0}}

# Galaxy

## Position

In [20]:
# 1.0" North
gal_coord = frb_coord.directional_offset_by(0.*units.deg, 1.0*units.arcsec)
gal_coord, gal_coord.separation(frb_coord).to('arcsec')

(<SkyCoord (ICRS): (ra, dec) in deg
     (326.10522917, -40.89975)>,
 <Angle 1. arcsec>)

## Size

In [21]:
small_gal_size = np.array([1.]) # arcsec
tiny_gal_size = np.array([0.2]) # arcsec
box_hwidth = 50.

# Calculate

## Small galaxy (1")

In [24]:
for step_size in [0.01, 0.1, 0.25, 0.5, 1., 5.]:
    L_wx, p_wOi, grid_p, p_xOis = bayesian.px_Oi_fixedgrid(box_hwidth, Path.localiz, np.array([gal_coord]),
                    small_gal_size, theta_prior, step_size=step_size, return_debug=True)
    print('================================================')
    print(f'step: {step_size}')
    print(f'L_wx: {np.sum(L_wx) * step_size**2}')
    print(f'p_wO: {np.sum(p_wOi) * step_size**2}')
    print(f'p_xO: {p_xOis}')
    print(f'p_xO_corr: {p_xOis/np.sum(p_wOi)/step_size**2}')

step: 0.01
L_wx: 0.9998000103183573
p_wO: 0.9997974697127257
p_xO: 0.005678161376930544
p_xO_corr: 0.005679311609542346
step: 0.1
L_wx: 0.998001000318597
p_wO: 0.9980224620716001
p_xO: 0.005678195745918647
p_xO_corr: 0.005689446842841982
step: 0.25
L_wx: 0.995006250316858
p_wO: 0.9949216063486281
p_xO: 0.0056773558179761905
p_xO_corr: 0.005706334832562478
step: 0.5
L_wx: 0.9900250003138782
p_wO: 0.9886994904374402
p_xO: 0.0056697174162953674
p_xO_corr: 0.005734520419128423
step: 1.0
L_wx: 0.9801000003123983
p_wO: 0.9702169732356216
p_xO: 0.00561310714089473
p_xO_corr: 0.005785414289522599
step: 5.0
L_wx: 0.9024999341095425
p_wO: 0.457497692474131
p_xO: 0.0024463480206865554
p_xO_corr: 0.005347235758626002


# Tiny galaxy (0.2")

In [27]:
for step_size in [0.01, 0.05, 0.1, 0.25, 0.5, 1.]:#, 5.]:
    L_wx, p_wOi, grid_p, p_xOis = bayesian.px_Oi_fixedgrid(box_hwidth, Path.localiz, np.array([gal_coord]),
                    tiny_gal_size, theta_prior, step_size=step_size, return_debug=True)
    print('================================================')
    print(f'step: {step_size}')
    print(f'L_wx: {np.sum(L_wx) * step_size**2}')
    print(f'p_wO: {np.sum(p_wOi) * step_size**2}')
    print(f'p_xO: {p_xOis}')
    print(f'p_xO_corr: {p_xOis/np.sum(p_wOi)/step_size**2}')

step: 0.01
L_wx: 0.9998000103183573
p_wO: 0.9998049191069248
p_xO: 0.006214941133239931
p_xO_corr: 0.006216153786072012
step: 0.05
L_wx: 0.9990002503180448
p_wO: 0.998734557460303
p_xO: 0.00621326369200773
p_xO_corr: 0.00622113618237816
step: 0.1
L_wx: 0.998001000318597
p_wO: 0.9963147603287327
p_xO: 0.00620441779104315
p_xO_corr: 0.006227367131443492
step: 0.25
L_wx: 0.995006250316858
p_wO: 0.9775937336096671
p_xO: 0.006105245905153129
p_xO_corr: 0.006245177004776943
step: 0.5
L_wx: 0.9900250003138782
p_wO: 0.8572364100422724
p_xO: 0.005378097137839817
p_xO_corr: 0.006273761910759962
step: 1.0
L_wx: 0.9801000003123983
p_wO: 0.455664596376793
p_xO: 0.002873215741084444
p_xO_corr: 0.006305549660717017
